In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
%cd /content/drive/MyDrive/Symba

/content/drive/MyDrive/Symba


In [ ]:
!git clone https://github.com/ML4SCI/SYMBA.git

Cloning into 'SYMBA'...
remote: Enumerating objects: 825, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 825 (delta 60), reused 44 (delta 44), pack-reused 694 (from 1)
Receiving objects: 100% (825/825), 1.91 MiB | 5.98 MiB/s, done.
Resolving deltas: 100% (339/339), done.
Updating files: 100% (362/362), done.


In [2]:
import math, random, re
from typing import Callable, List, Dict
import numpy as np

def sinc(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    m = np.abs(z) < 1e-12
    out[m] = 1.0
    out[~m] = np.sin(z[~m]) / z[~m]
    return out

def lorentz(z, s):
    return 1.0 / (1.0 + (z/s)**2)

def safe_log_pos(u):
    return np.log1p(np.asarray(u)**2)

def safe_sqrt_pos(u):
    return np.sqrt(1.0 + np.asarray(u)**2)

def gaussian(x, mu, alpha):
    return np.exp(-alpha*(np.asarray(x)-mu)**2)

def poly_eval(x, coeffs):
    x = np.asarray(x, dtype=float)
    y = 0.0
    for c in reversed(coeffs):
        y = y*x + c
    return y

SAFE_NS = {
    'sin': np.sin,
    'cos': np.cos,
    'sinh': np.sinh,
    'cosh': np.cosh,
    'tanh': np.tanh,
    'exp': np.exp,
    'log': np.log,
    'sqrt': np.sqrt,
    'abs': np.abs,
    'sinc': sinc,
    'lorentz': lorentz,
    'gaussian': gaussian,
    'poly': poly_eval,
    'pi': math.pi
}

class Expr:
    def __init__(self, fn: Callable, s: str):
        self.fn = fn
        self.s = s

    def eval(self, x):
        return self.fn(x)

def gen_poly_component():
    deg = random.randint(1, 4)
    coeffs = [random.uniform(-2, 2) for _ in range(deg+1)]
    def f(x): return poly_eval(x, coeffs)
    terms = []
    for i,c in enumerate(coeffs):
        if abs(c) < 1e-12: continue
        terms.append(f"{c:.6g}" if i==0 else f"{c:.6g}*x" if i==1 else f"{c:.6g}*x**{i}")
    s = " + ".join(terms) if terms else "0.0"
    return Expr(f, s)

def gen_trig_component():
    A = random.uniform(0.2, 5.0)
    k = random.randint(1, 8)
    phi = random.uniform(-math.pi, math.pi)
    which = random.choice(['sin', 'cos'])
    def f(x):
        if np is None:
            return A*(math.sin(k*x+phi) if which=='sin' else math.cos(k*x+phi))
        return A*(np.sin(k*x+phi) if which=='sin' else np.cos(k*x+phi))
    return Expr(f, f"{A:.6g}*{which}({k}*x + {phi:.6g})")

def gen_damped_trig_component():
    base = gen_trig_component()
    if random.random() < 0.5:
        alpha = random.uniform(0.1, 2.5); mu = random.uniform(-math.pi/2, math.pi/2)
        def f(x): return base.fn(x) * gaussian(x, mu, alpha)
        return Expr(f, f"({base.s})*exp(-{alpha:.6g}*(x-{mu:.6g})**2)")
    s_param = random.uniform(0.5, 3.0)
    def f(x): return base.fn(x) * lorentz(x, s_param)
    return Expr(f, f"({base.s})*lorentz(x, {s_param:.6g})")

def gen_tanh_component():
    A = random.uniform(0.2, 5.0); k = random.uniform(0.5, 3.0); b = random.uniform(-3.0, 3.0)
    def f(x): return A*(np.tanh(k*x+b))
    return Expr(f, f"{A:.6g}*tanh({k:.6g}*x + {b:.6g})")

def gen_sinh_component():
    A = random.uniform(0.1, 2.0); beta = random.uniform(0.2, 1.0); alpha = random.uniform(0.05, 0.5)
    def f(x): return A*((np.sinh(beta*x))*gaussian(x, 0.0, alpha))
    return Expr(f, f"{A:.6g}*sinh({beta:.6g}*x)*exp(-{alpha:.6g}*x**2)")

def gen_cosh_component():
    A = random.uniform(0.1, 3.0); beta = random.uniform(0.2, 1.0); alpha = random.uniform(0.05, 0.5)
    def f(x): return A*((np.cosh(beta*x))*gaussian(x, 0.0, alpha))
    return Expr(f, f"{A:.6g}*cosh({beta:.6g}*x)*exp(-{alpha:.6g}*x**2)")

def gen_log_component():
    A = random.uniform(0.2, 3.0); p = gen_poly_component()
    def f(x): return A*safe_log_pos(p.fn(x))
    return Expr(f, f"{A:.6g}*log(1 + ({p.s})**2)")

def gen_sqrt_component():
    p = gen_poly_component()
    def f(x): return safe_sqrt_pos(p.fn(x))
    return Expr(f, f"sqrt(1 + ({p.s})**2)")

def gen_rational_component():
    A = random.uniform(0.2, 4.0); p = gen_poly_component(); q = gen_poly_component()
    def f(x):
        num = p.fn(x); den = 1.0 + (q.fn(x))**2
        return A*num/den
    return Expr(f, f"{A:.6g}*(({p.s})/(1 + ({q.s})**2))")

def gen_sinc_component():
    A = random.uniform(0.2, 3.0); k = random.uniform(0.5, 5.0); phi = random.uniform(-2.0, 2.0)
    def f(x): return A*sinc(k*x + phi)
    return Expr(f, f"{A:.6g}*sinc({k:.6g}*x + {phi:.6g})")

def gen_gaussian_component():
    A = random.uniform(0.2, 5.0); mu = random.uniform(-math.pi/2, math.pi/2); alpha = random.uniform(0.1, 3.0)
    def f(x): return A*gaussian(x, mu, alpha)
    return Expr(f, f"{A:.6g}*exp(-{alpha:.6g}*(x-{mu:.6g})**2)")

def gen_pow_component():
    base = random.choice([gen_poly_component, gen_trig_component, gen_damped_trig_component,
                          gen_gaussian_component, gen_sinc_component, gen_rational_component])()
    n = random.randint(2, 4)
    def f(x): return base.fn(x)**n
    return Expr(f, f"({base.s})**{n}")

def sum_components(a: Expr, b: Expr):
    def f(x): return a.fn(x) + b.fn(x)
    return Expr(f, f"({a.s}) + ({b.s})")

def product_components(a: Expr, b: Expr):
    def f(x): return a.fn(x) * b.fn(x)
    return Expr(f, f"({a.s})*({b.s})")

COMPONENT_BUILDERS = [
    gen_poly_component, gen_trig_component, gen_damped_trig_component, gen_tanh_component,
    gen_sinh_component, gen_cosh_component, gen_log_component, gen_sqrt_component,
    gen_rational_component, gen_sinc_component, gen_gaussian_component, gen_pow_component
]

BANNED_NESTINGS = [
    r'sin\s*\(\s*sin', r'sin\s*\(\s*cos', r'cos\s*\(\s*sin', r'cos\s*\(\s*cos',
    r'sin\s*\(\s*tanh', r'cos\s*\(\s*tanh', r'sin\s*\(\s*sinh', r'cos\s*\(\s*sinh',
    r'exp\s*\(\s*exp', r'log\s*\(\s*log', r'sqrt\s*\(\s*sqrt'
]

def violates_nesting(expr_str: str) -> bool:
    s = expr_str.replace(' ', '')
    return any(re.search(p, s) for p in BANNED_NESTINGS)

def gen_expression() -> Expr:
    mode = random.choices(['single','sum2','prod2','sum3'], [0.35,0.35,0.2,0.1])[0]
    a = random.choice(COMPONENT_BUILDERS)()
    if mode == 'single': return a
    b = random.choice(COMPONENT_BUILDERS)()
    if mode == 'sum2':   return sum_components(a, b)
    if mode == 'prod2':  return product_components(a, b)
    c = random.choice(COMPONENT_BUILDERS)()
    return sum_components(sum_components(a,b), c)

def make_domain(n=2048):
    return np.linspace(-math.pi, math.pi, n)

def is_finite_array(y) -> bool:
    return np.isfinite(np.asarray(y)).all()

def is_plausible(expr: Expr, xgrid) -> bool:
    try:
        y = expr.eval(xgrid)
        if not is_finite_array(y): return False
        dy = np.gradient(y, xgrid)
        if not np.isfinite(dy).all(): return False
        amp = float(np.max(np.abs(y)))
        dmax = float(np.max(np.abs(dy)))
        if dmax > 1e4 and amp > 1.0: return False
        return True
    except Exception:
        return False

def generate_dataset(target=100, max_tries=5000, seed=0, grid_n=2048) -> List[Dict[str,str]]:
    random.seed(seed)
    np.random.seed(seed)
    xgrid = make_domain(grid_n)
    seen, out, tries = set(), [], 0
    while len(out) < target and tries < max_tries:
        tries += 1
        expr = gen_expression()
        if expr.s in seen: continue
        if violates_nesting(expr.s): continue
        if not is_plausible(expr, xgrid): continue
        seen.add(expr.s)
        out.append({'expr': expr.s})
    return out

def make_callable_from_string(expr_str: str) -> Callable:
    """Compile a safe callable f(x) for the expression string."""
    def f(x):
        env = dict(SAFE_NS); env['x'] = x
        return eval(expr_str, {"__builtins__": {}}, env)
    return f

In [ ]:
# Generate 2M candidates, keep those finite and plausible
ds = generate_dataset(target=2_000_000, max_tries=2_500_000, seed=0)

# Get just the expressions
expressions = [row['expr'] for row in ds]

In [ ]:
resolution = 1024
x = np.linspace(-np.pi, np.pi, resolution)

y = np.zeros((len(expressions), resolution), dtype=float)
for i, expr in enumerate(expressions):
    f = make_callable_from_string(expr)
    y[i,:] = f(x)

In [ ]:
import pickle as pkl

with open('data/synthetic.pkl', 'wb') as f:
    pkl.dump(expressions, f)

In [ ]:
with open('/content/drive/MyDrive/SYMBA/SYMBA/values.pkl', 'wb') as f:
    pkl.dump(y, f)

In [ ]:
print(expressions[0:10])

['(0.313356*log(1 + (-0.380263 + 1.13519*x + -0.786749*x**2 + -0.0936122*x**3 + 0.333528*x**4)**2))*(2.6225*tanh(1.20459*x + 1.53483))', '(1.97579*sinh(0.626051*x)*exp(-0.367328*x**2)) + (2.72606*sinc(1.89566*x + 0.919327))', '(1.78495*exp(-1.12599*(x-0.18791)**2))*(1.87157*cosh(0.930409*x)*exp(-0.484973*x**2))', '(1.18987*((-1.62691 + 1.36036*x)/(1 + (0.447588 + 1.31225*x)**2))) + (0.807342*cosh(0.460163*x)*exp(-0.441712*x**2))', '0.820771*sinc(4.11505*x + -1.43002)', '(0.58614*cos(6*x + 2.35674)) + (1.64694*cos(5*x + 1.29786))', '(0.972167*((-0.220044 + 0.385147*x + -0.460395*x**2 + 0.302604*x**3)/(1 + (-1.26453 + 1.28587*x + -1.86811*x**2 + 1.9252*x**3)**2)))*(1.00541*sinh(0.271859*x)*exp(-0.390922*x**2))', '(1.59269 + 1.69233*x)*(2.79729*((0.0871604 + -1.05799*x + -1.13919*x**2 + 0.717897*x**3)/(1 + (0.31878 + -0.197748*x + 0.640982*x**2 + 1.98503*x**3 + 1.66776*x**4)**2)))', '(1.75676*cos(2*x + -0.0851736))*(1.36657*tanh(2.32872*x + -2.29719))', '(1.79617*sin(1*x + -2.50946))*lore